# 🔍 Pipeline phát hiện Teencode tiếng Việt bằng ViSoBERT (v11)

Cập nhật v10: WeightedTrainer (class weights + label smoothing), augmentation câu teencode ngắn, rescue threshold động theo độ dài từ, MIN_CONFIDENCE hạ 0.5→0.4, EPOCHS tăng 8→10.

- **[v11] MAX_LEN 128→192**: đồng bộ với PhoBERT, bắt cụm gen-Z dài
- **[v11] Đánh giá chuẩn hóa**: đồng bộ tiêu chí partial_match ≥2/3 với PhoBERT


## 1. Cài đặt thư viện & Cấu hình

In [1]:
# Cài đặt thư viện cần thiết
# Lưu ý: trên Kaggle GPU, "torch" đã được cài sẵn đúng với driver CUDA của máy ảo,
# nên KHÔNG cài đè torch qua pip (dễ làm mất tương thích CUDA) — chỉ cài các gói còn thiếu.
import importlib, subprocess, sys

required = ["python-dotenv", "sentencepiece", "seqeval", "evaluate", "openpyxl", "transformers", "datasets", "accelerate"]
# Lưu ý: ViSoBERT (kiến trúc XLM-R) dùng tokenizer SentencePiece -> cần gói "sentencepiece".
# Không cần "underthesea" nữa vì ViSoBERT không yêu cầu tách từ trước khi tokenize.
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print("Đang cài đặt các thư viện còn thiếu:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("Đã có đủ thư viện cần thiết.")

import torch
assert torch.cuda.is_available(), (
    "Không tìm thấy GPU! Trên Kaggle: vào Settings (bên phải) -> Accelerator -> chọn GPU T4 x2 hoặc P100, "
    "rồi chạy lại notebook."
)
print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda} | torch: {torch.__version__}")


Đang cài đặt các thư viện còn thiếu: ['seqeval', 'evaluate']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
GPU: Tesla T4 | CUDA: 12.8 | torch: 2.10.0+cu128


In [2]:
import os
import re
import json
import random
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Nạp biến môi trường từ file .env (nếu có) — KHÔNG commit .env lên Git.
# Xem .env.example để biết các biến có thể override (đường dẫn dữ liệu, ...).
# Nếu không có .env (vd. chạy trên Kaggle), pipeline tự dùng giá trị mặc định
# (đường dẫn /kaggle/input/...) như trước đây — không cần chỉnh sửa gì thêm.
# ---------------------------------------------------------------------------
from dotenv import load_dotenv
load_dotenv()

# ---------------------------------------------------------------------------
# Tự động dò đường dẫn dữ liệu đầu vào
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# CẤU HÌNH CHUNG
# ---------------------------------------------------------------------------
CONFIG = {
    "TEENCODE_PATH": os.getenv(
        "TEENCODE_PATH",
        "/kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_Teencode.xlsx",
    ),
    "CFS_PATH": os.getenv(
        "CFS_PATH",
        "/kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_CFS.xlsx",
    ),

    "CFS_SAMPLE_SIZE": (
        int(os.getenv("CFS_SAMPLE_SIZE"))
        if os.getenv("CFS_SAMPLE_SIZE")
        else None
    ),
    "RANDOM_SEED": int(os.getenv("RANDOM_SEED", 42)),

    "MODEL_NAME": "uitnlp/visobert",
    "MAX_LEN": 192,         # [v11] tăng 128→192 đồng bộ PhoBERT, bắt cụm gen-Z dài
    "LABELS": ["O", "B-TEENCODE", "I-TEENCODE"],

    # [v9] Tăng EPOCHS 8→10, thêm WARMUP_RATIO và WEIGHT_DECAY
    "EPOCHS": 10,
    "LEARNING_RATE": 2e-5,
    "WARMUP_RATIO": 0.1,           # warmup 10% đầu để ổn định gradient
    "WEIGHT_DECAY": 0.01,
    "BATCH_SIZE": 32,
    "EVAL_BATCH_SIZE": 64,
    "GRADIENT_ACCUMULATION_STEPS": 1,
    "FP16": torch.cuda.is_available(),
    "DATALOADER_NUM_WORKERS": 2,
    "EARLY_STOPPING_PATIENCE": 3,  # tăng 2→3 vì training dài hơn
    "TRAIN_RATIO": 0.8,
    "VAL_RATIO": 0.1,
    "TEST_RATIO": 0.1,

    # [v11] MIN_CONFIDENCE=0.4 — mốc chung đồng bộ 3 thuật toán
    "MIN_CONFIDENCE": 0.4,
    # [v11] RESCUE_MIN_CONFIDENCE=0.10 — mốc chung đồng bộ 3 thuật toán
    "RESCUE_MIN_CONFIDENCE": 0.10,

    # [v9] Augmentation: oversample câu chứa teencode ngắn
    "AUGMENT_TARGET_RATIO": 0.45,

    "MODEL_DIR": os.getenv(
        "MODEL_DIR",
        "/kaggle/working/models/visobert_teencode_detector" if os.path.isdir("/kaggle/working")
        else "models/visobert_teencode_detector",
    ),
    "OUTPUT_DIR": os.getenv(
        "OUTPUT_DIR",
        "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs",
    ),
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["MODEL_DIR"], exist_ok=True)

random.seed(CONFIG["RANDOM_SEED"])
np.random.seed(CONFIG["RANDOM_SEED"])
torch.manual_seed(CONFIG["RANDOM_SEED"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["RANDOM_SEED"])
    torch.backends.cudnn.benchmark = True

LABEL2ID = {l: i for i, l in enumerate(CONFIG["LABELS"])}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

print("Cấu hình:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


Cấu hình:
  TEENCODE_PATH: /kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_Teencode.xlsx
  CFS_PATH: /kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_CFS.xlsx
  CFS_SAMPLE_SIZE: None
  RANDOM_SEED: 42
  MODEL_NAME: uitnlp/visobert
  MAX_LEN: 192
  LABELS: ['O', 'B-TEENCODE', 'I-TEENCODE']
  EPOCHS: 10
  LEARNING_RATE: 2e-05
  WARMUP_RATIO: 0.1
  WEIGHT_DECAY: 0.01
  BATCH_SIZE: 32
  EVAL_BATCH_SIZE: 64
  GRADIENT_ACCUMULATION_STEPS: 1
  FP16: True
  DATALOADER_NUM_WORKERS: 2
  EARLY_STOPPING_PATIENCE: 3
  TRAIN_RATIO: 0.8
  VAL_RATIO: 0.1
  TEST_RATIO: 0.1
  MIN_CONFIDENCE: 0.4
  RESCUE_MIN_CONFIDENCE: 0.1
  AUGMENT_TARGET_RATIO: 0.45
  MODEL_DIR: /kaggle/working/models/phobert_teencode_detector
  OUTPUT_DIR: /kaggle/working/outputs


## 2. Tải dữ liệu

- `teen_df`: từ điển teencode (sheet `Dataset chính`) — dùng để xây lexicon gán nhãn yếu (weak/distant supervision) và để so sánh "teencode đã biết" ở Giai đoạn 8.
- `cfs_df`: dữ liệu CFS thực tế (sheet `data_clean`) — dùng để xây dataset train + để đánh giá + để chạy dự đoán hàng loạt.

In [3]:
teen_df = pd.read_excel(CONFIG["TEENCODE_PATH"], sheet_name="Teencode")
cfs_df = pd.read_excel(CONFIG["CFS_PATH"], sheet_name="CFS")

print("Teencode dataset:", teen_df.shape)
print("CFS dataset:", cfs_df.shape)

teen_df.head(3)


Teencode dataset: (1739, 8)
CFS dataset: (21473, 5)


,STT,Teencode,Nghĩa,Dạng chuẩn hóa,Loại teencode,Ngữ cảnh,Nhạy cảm,Cần ngữ cảnh
0,1,1314,một đời một kiếp,một đời một kiếp,Ký hiệu/Số,Thường dùng trong tình yêu để diễn đạt ý 'một ...,0,0
1,2,1m,một mình,một mình,Ký hiệu/Số,"Thường dùng trong tâm trạng, hoạt động cá nhân...",0,0
2,3,3in1,3 trong 1,3 trong 1,Ký hiệu/Số,Thường dùng trong quảng cáo sản phẩm để diễn đ...,0,0


In [4]:
cfs_df.head(3)


,STT,Văn bản gốc (CFS),Độ dài (ký tự),Độ dài (từ),Nhãn (gốc)
0,1,UTC2 đúng thật sự là Đường đến thành công,41,9,"""UTC2"""
1,2,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,39,10,"""KTX"""
2,3,21 tuổi là lớn rồi nha Hậu trường tối qua,41,10,NaN


## 3. Giai đoạn 2: Tiền xử lý dữ liệu

### 3.1 Làm sạch text
Áp dụng theo đúng yêu cầu: chuẩn hóa Unicode, lowercase, loại bỏ ký tự dư thừa, chuẩn hóa khoảng trắng, xử lý emoji/ký tự đặc biệt.

In [5]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"   # symbols & pictographs, emoticons, transport...
    "\U00002700-\U000027BF"   # dingbats
    "\U0001F1E6-\U0001F1FF"   # flags
    "\U00002600-\U000026FF"   # misc symbols
    "\U0001F000-\U0001F0FF"   # mahjong/playing cards
    "]+",
    flags=re.UNICODE,
)


def normalize_unicode(text: str) -> str:
    '''Chuẩn hóa Unicode về dạng NFC (tránh lỗi do tổ hợp dấu tiếng Việt khác nhau).'''
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFC", text)


def clean_text(text: str) -> str:
    '''Làm sạch text theo Giai đoạn 2.1:
    - Chuẩn hóa Unicode
    - Lowercase
    - Loại bỏ ký tự dư thừa (lặp ký tự, lặp dấu câu)
    - Xử lý emoji / ký tự đặc biệt
    - Chuẩn hóa khoảng trắng
    '''
    text = normalize_unicode(text)
    text = text.lower()

    # Loại bỏ URL
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Xử lý emoji -> khoảng trắng
    text = EMOJI_PATTERN.sub(" ", text)

    # Loại bỏ lặp ký tự kéo dài quá mức (vd: "lunnnnn" -> "lunn", "đẹppppp" -> "đẹpp")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # Chuẩn hóa lặp dấu câu (vd: "!!!" -> "!", "???" -> "?")
    text = re.sub(r"([!?.,])\1+", r"\1", text)

    # Loại bỏ ký tự đặc biệt không cần thiết, giữ chữ/số/dấu câu cơ bản tiếng Việt
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹà-ỹ\s.,!?'\"-]", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Demo
demo_text = "Tuiii    ko  bt hôm nay đi học ko!!! 😂😂😂 xem link http://abc.com nha~~~"
print("Input :", demo_text)
print("Output:", clean_text(demo_text))


Input : Tuiii    ko  bt hôm nay đi học ko!!! 😂😂😂 xem link http://abc.com nha~~~
Output: tuii ko bt hôm nay đi học ko! xem link nha


### 3.2 Tokenization (tách âm tiết)
ViSoBERT được pretrain trực tiếp trên văn bản tiếng Việt **không qua tách từ** (khác PhoBERT). Vì vậy ở đây ta chỉ cần tách câu thành các **âm tiết** bằng khoảng trắng (đã chuẩn hoá ở bước `clean_text`) — không dùng `underthesea.word_tokenize` nữa.


In [6]:
def tokenize(text: str):
    '''Tokenize văn bản tiếng Việt đã làm sạch thành list âm tiết (ViSoBERT không cần tách từ
    như PhoBERT — chỉ cần tách theo khoảng trắng).'''
    cleaned = clean_text(text)
    if not cleaned:
        return []
    return cleaned.split()


# Demo đúng với ví dụ trong kế hoạch
example = "tui ko bt hôm nay đi học ko"
tokens = tokenize(example)
print("Input :", example)
print("Output:", " | ".join(tokens))


Input : tui ko bt hôm nay đi học ko
Output: tui | ko | bt | hôm | nay | đi | học | ko


## 4. Giai đoạn 3: Xây dựng dataset train ViSoBERT

**Cách gán nhãn BIO (weak/distant supervision):** Ta xây một **lexicon** từ cột `Teencode` của bộ từ điển teencode (1218 mục). Với mỗi câu CFS, sau khi tokenize, ta dò từng cụm token liên tiếp (n-gram, tối đa bằng độ dài mục từ điển dài nhất) xem có khớp với một mục trong lexicon không:

- Token đầu tiên của cụm khớp → `B-TEENCODE`
- Các token tiếp theo trong cụm → `I-TEENCODE`
- Token không khớp → `O`

> ⚠️ Đây là gán nhãn tự động (weak label) dựa trên từ điển — sẽ có nhiễu với các từ ngắn đa nghĩa (vd: "ai" vừa là đại từ "ai" vừa là viết tắt "AI"). Đây là hạn chế cố hữu của distant supervision; có thể cải thiện sau bằng việc rà soát/gán nhãn thủ công thêm (xem Giai đoạn 9).

In [7]:
def normalize_lexicon_entry(entry: str):
    '''Chuẩn hóa 1 mục teencode trong từ điển về tuple âm tiết (dùng đúng hàm tokenize() ở trên,
    để đảm bảo lexicon và câu CFS được tách theo cùng 1 cách).'''
    return tuple(tokenize(str(entry)))


# Xây lexicon từ bộ từ điển teencode
teen_df["Teencode"] = teen_df["Teencode"].astype(str)
lexicon_entries = set()
for v in teen_df["Teencode"]:
    toks = normalize_lexicon_entry(v)
    if 1 <= len(toks) <= 6:
        lexicon_entries.add(toks)

MAX_LEX_LEN = max(len(t) for t in lexicon_entries)
LEXICON_BY_LEN = defaultdict(set)
for t in lexicon_entries:
    LEXICON_BY_LEN[len(t)].add(t)

# Tập "teencode đã biết" ở dạng text (dùng lại cho Giai đoạn 8)
KNOWN_TEENCODE_SET = {" ".join(t) for t in lexicon_entries}

# Một cụm dự đoán dài hơn MAX_LEX_LEN + 1 token gần như chắc chắn là model bị "lan" nhãn
# sang các từ tiếng Việt bình thường xung quanh -> sẽ được hậu kiểm tra lại bằng từ điển
# ở bước dự đoán (xem hàm predict_teencode_spans ở Giai đoạn 6).
CONFIG["MAX_VALID_SPAN_LEN"] = MAX_LEX_LEN + 1

print(f"Số mục lexicon: {len(lexicon_entries)} | độ dài token tối đa: {MAX_LEX_LEN}")
print(f"MAX_VALID_SPAN_LEN (ngưỡng cụm hợp lệ khi dự đoán): {CONFIG['MAX_VALID_SPAN_LEN']}")

# ─────────────────────────────────────────────────────────────────────────────
# 🆕 PHÂN LOẠI LEXICON: NHÓM A (an toàn) vs NHÓM B (nhập nhằng)
# ─────────────────────────────────────────────────────────────────────────────
# Nhóm B = các entry có require_context == 1 trong dataset (độ dài ≤ 2 ký tự,
# dễ match nhầm vào từ tiếng Việt chuẩn như "a", "c", "e", "m", "t", "h"...).
# Nhóm A = tất cả còn lại — khi xuất hiện gần như chắc chắn là teencode.
#
# Mục tiêu: rescue pass CHỈ áp dụng mạnh (không cần ngưỡng confidence cao)
# với Nhóm A; còn Nhóm B vẫn để model quyết định (tránh sinh FP).

require_context_set = set()
if "Cần ngữ cảnh" in teen_df.columns:
    rc_mask = teen_df["Cần ngữ cảnh"] == 1
    for v in teen_df.loc[rc_mask, "Teencode"]:
        toks = normalize_lexicon_entry(v)
        if toks:
            require_context_set.add(toks)

# Nhóm A: lexicon entry KHÔNG thuộc require_context (an toàn để rescue)
LEXICON_GROUP_A = {t for t in lexicon_entries if t not in require_context_set}
# Nhóm B: lexicon entry thuộc require_context (nhập nhằng — KHÔNG rescue)
LEXICON_GROUP_B = require_context_set & lexicon_entries

# Build lookup nhanh theo độ dài cho Nhóm A
LEXICON_A_BY_LEN = defaultdict(set)
for t in LEXICON_GROUP_A:
    LEXICON_A_BY_LEN[len(t)].add(t)

print(f"\n📂 Phân loại lexicon:")
print(f"  Nhóm A (an toàn  — rescue được): {len(LEXICON_GROUP_A)} entries")
print(f"  Nhóm B (nhập nhằng — skip rescue): {len(LEXICON_GROUP_B)} entries")
top_b = sorted([' '.join(t) for t in list(LEXICON_GROUP_B)[:20]])
print(f"  Ví dụ Nhóm B: {top_b}")


Số mục lexicon: 1617 | độ dài token tối đa: 6
MAX_VALID_SPAN_LEN (ngưỡng cụm hợp lệ khi dự đoán): 7

📂 Phân loại lexicon:
  Nhóm A (an toàn  — rescue được): 1488 entries
  Nhóm B (nhập nhằng — skip rescue): 129 entries
  Ví dụ Nhóm B: ['ace', 'aochính', 'csvhvn', 'cđs', 'hd', 'huit', 'hv', 'j', 'km', 'lkers', 'nckhkltn', 'ou', 'pgp', 'pgpb', 'pltt', 'pvc', 'tb', 'tdtm', 'ultr', 'vac']


In [8]:
def bio_label_sentence(text: str):
    """Tokenize 1 câu và gán nhãn BIO theo lexicon teencode.
    [v9] Thêm fallback lowercase-strip để bắt token ngắn bị tokenizer normalize.
    Trả về (tokens, labels).
    """
    tokens = tokenize(text)
    n = len(tokens)
    labels = ["O"] * n
    i = 0
    while i < n:
        matched_len = 0
        max_try = min(MAX_LEX_LEN, n - i)
        for L in range(max_try, 0, -1):
            candidate = tuple(tokens[i:i + L])
            if candidate in LEXICON_BY_LEN.get(L, set()):
                matched_len = L
                break
        if matched_len > 0:
            labels[i] = "B-TEENCODE"
            for k in range(1, matched_len):
                labels[i + k] = "I-TEENCODE"
            i += matched_len
        else:
            # [v9] Fallback: thử match token đơn ở dạng lowercase thuần
            # (xử lý trường hợp tokenizer giữ hoa thị hoặc dấu câu dính vào)
            tok_norm = re.sub(r'[.,!?\'"]', '', tokens[i].lower())
            if tok_norm and (tok_norm,) in LEXICON_BY_LEN.get(1, set()):
                labels[i] = "B-TEENCODE"
            i += 1
    return tokens, labels


# [v9] Augmentation: oversample câu chứa teencode ngắn (≤2 ký tự)
# Mục tiêu: nâng tỉ lệ câu có teencode lên ~AUGMENT_TARGET_RATIO
# để giảm bias của model về nhãn O.
def augment_bio_records(bio_records, target_ratio=None):
    if target_ratio is None:
        target_ratio = CONFIG.get("AUGMENT_TARGET_RATIO", 0.45)

    SHORT_TC = {t for t in lexicon_entries if len(" ".join(t).replace(" ", "")) <= 2}

    has_short, no_short = [], []
    for rec in bio_records:
        toks = rec["tokens"]
        labs = rec["labels"]
        hit = any(
            tuple(toks[i:i+1]) in SHORT_TC
            for i, l in enumerate(labs) if l == "B-TEENCODE"
        )
        (has_short if hit else no_short).append(rec)

    current_ratio = len(has_short) / max(len(bio_records), 1)
    if current_ratio >= target_ratio or len(has_short) == 0:
        print(f"Augment: skip (tỉ lệ hiện tại {current_ratio:.1%} >= mục tiêu {target_ratio:.1%})")
        return bio_records

    n_extra = int(
        (target_ratio * len(no_short) - (1 - target_ratio) * len(has_short))
        / max(1 - target_ratio, 1e-6)
    )
    n_extra = min(n_extra, len(has_short) * 4)
    extra = random.choices(has_short, k=n_extra)
    result = bio_records + extra
    random.shuffle(result)
    new_ratio = (len(has_short) + n_extra) / len(result)
    print(f"Augment: {len(bio_records)} → {len(result)} câu "
          f"| tỉ lệ short-teencode: {current_ratio:.1%} → {new_ratio:.1%}")
    return result


# Demo
demo = "mình ko bt nha"
toks, labs = bio_label_sentence(demo)
for t, l in zip(toks, labs):
    print(f"{t}\t{l}")


mình	O
ko	B-TEENCODE
bt	B-TEENCODE
nha	O


In [9]:
# Lấy mẫu CFS_SAMPLE_SIZE dòng để xây dataset
cfs_text_col = "Văn bản gốc (CFS)"
valid_cfs = cfs_df[cfs_df[cfs_text_col].notna()].copy()

sample_size = CONFIG["CFS_SAMPLE_SIZE"] or len(valid_cfs)
sample_size = min(sample_size, len(valid_cfs))

cfs_sample = valid_cfs.sample(n=sample_size, random_state=CONFIG["RANDOM_SEED"]).reset_index(drop=True)
print(f"Số dòng CFS dùng để xây dataset BIO: {len(cfs_sample)}")

bio_records = []
for idx, row in cfs_sample.iterrows():
    tokens, labels = bio_label_sentence(row[cfs_text_col])
    if len(tokens) == 0:
        continue
    bio_records.append({
        "sent_id": idx,
        "tokens": tokens,
        "labels": labels,
        "has_teencode": any(l != "O" for l in labels),
    })

# [v9] Augment trước khi build DataFrame
bio_records = augment_bio_records(bio_records, target_ratio=CONFIG.get("AUGMENT_TARGET_RATIO", 0.45))

bio_df = pd.DataFrame(bio_records)
print(f"Số câu hợp lệ sau tokenize + augment: {len(bio_df)}")
print(f"Tỉ lệ câu có chứa teencode (theo lexicon): {bio_df['has_teencode'].mean():.2%}")
bio_df.head(3)


Số dòng CFS dùng để xây dataset BIO: 21473
Augment: skip (tỉ lệ hiện tại 58.6% >= mục tiêu 45.0%)
Số câu hợp lệ sau tokenize + augment: 21473
Tỉ lệ câu có chứa teencode (theo lexicon): 80.22%


,sent_id,tokens,labels,has_teencode
0,0,"[em, chào, mọi, người, ạ,, lại, là, em,, bạn, ...","[O, O, O, O, O, O, O, O, O, O, O, B-TEENCODE, ...",True
1,1,"[thấy, đề, lý, 10, năm, nay, dễ, hơn, năm, ngo...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",False
2,2,"[mình, muốn, pass, lại, lõi, cushion, lucenbas...","[O, O, B-TEENCODE, O, O, O, O, O, O, O, O, B-T...",True


In [10]:
# Chia Train/Validation/Test = 80/10/10
train_df, temp_df = train_test_split(
    bio_df, train_size=CONFIG["TRAIN_RATIO"], random_state=CONFIG["RANDOM_SEED"], shuffle=True
)
val_ratio_within_temp = CONFIG["VAL_RATIO"] / (CONFIG["VAL_RATIO"] + CONFIG["TEST_RATIO"])
val_df, test_df = train_test_split(
    temp_df, train_size=val_ratio_within_temp, random_state=CONFIG["RANDOM_SEED"], shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")


Train: 17178 | Validation: 2147 | Test: 2148


## 5. Giai đoạn 4: Train ViSoBERT Token Classification

Pipeline: `Input Text → Tách âm tiết (khoảng trắng) → ViSoBERT Tokenizer → ViSoBERT Encoder → Classification Layer → O / B-TEENCODE / I-TEENCODE`

Cấu hình train theo kế hoạch: Train/Val/Test 80/10/10, Epoch 5-10, Learning rate 2e-5, Batch size 16, Optimizer AdamW.

In [11]:
import torch
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
import evaluate as hf_evaluate
try:
    seqeval_metric = hf_evaluate.load("seqeval")
except Exception:
    seqeval_metric = None

from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"], use_fast=False)
model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["MODEL_NAME"],
    num_labels=len(CONFIG["LABELS"]),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)
print("Đã tải ViSoBERT:", CONFIG["MODEL_NAME"])

# [v9] Tính class weights từ phân phối nhãn thực tế trong bio_records
all_labels_flat = [l for rec in bio_records for l in rec["labels"]]
total = len(all_labels_flat)
cnt_o = all_labels_flat.count("O")
cnt_b = all_labels_flat.count("B-TEENCODE")
cnt_i = all_labels_flat.count("I-TEENCODE")
w_o = min(total / (3 * max(cnt_o, 1)), 2.0)
w_b = min(total / (3 * max(cnt_b, 1)), 10.0)
w_i = min(total / (3 * max(cnt_i, 1)), 10.0)
CLASS_WEIGHTS = torch.tensor([w_o, w_b, w_i])
print(f"Class weights — O:{w_o:.3f}  B:{w_b:.3f}  I:{w_i:.3f}")

class WeightedTrainer(Trainer):
    """[v9] Custom Trainer với weighted cross-entropy + label smoothing.
    [fix] Dùng self._unwrap_model(model) để lấy config an toàn cả single-GPU lẫn DataParallel.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Unwrap DataParallel/FSDP để truy cập .config an toàn
        unwrapped = model.module if hasattr(model, "module") else model
        loss_fct = CrossEntropyLoss(
            weight=CLASS_WEIGHTS.to(logits.device),
            ignore_index=-100,
            label_smoothing=0.05,
        )
        loss = loss_fct(logits.view(-1, unwrapped.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


Device: cuda
  GPU: Tesla T4
  VRAM: 14.6 GB


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

Đã tải ViSoBERT: uitnlp/visobert
Class weights — O:0.363  B:4.151  I:10.000


In [12]:
def encode_with_labels(words, labels, tokenizer, max_len):
    '''Mã hóa 1 câu (đã tách thành list âm tiết) thành input_ids/attention_mask/labels, căn chỉnh
    nhãn BIO ở mức âm tiết sang nhãn ở mức subword-token của ViSoBERT.
    Subword đầu tiên của 1 âm tiết giữ nhãn của âm tiết đó; các subword tiếp theo gán -100 (bỏ qua khi tính loss).
    '''
    input_ids = [tokenizer.bos_token_id]
    label_ids = [-100]

    for word, lab in zip(words, labels):
        sub_tokens = tokenizer.tokenize(word)
        if len(sub_tokens) == 0:
            sub_tokens = [tokenizer.unk_token]
        sub_ids = tokenizer.convert_tokens_to_ids(sub_tokens)

        input_ids.extend(sub_ids)
        label_ids.append(LABEL2ID[lab])
        if len(sub_ids) > 1:
            label_ids.extend([-100] * (len(sub_ids) - 1))

    input_ids.append(tokenizer.eos_token_id)
    label_ids.append(-100)

    # Cắt nếu vượt max_len
    input_ids = input_ids[:max_len]
    label_ids = label_ids[:max_len]

    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        label_ids = label_ids + [-100] * pad_len
        attention_mask = attention_mask + [0] * pad_len

    return input_ids, attention_mask, label_ids


def build_hf_dataset(df, tokenizer, max_len):
    all_input_ids, all_attn, all_labels = [], [], []
    for _, row in df.iterrows():
        input_ids, attn, labels = encode_with_labels(row["tokens"], row["labels"], tokenizer, max_len)
        all_input_ids.append(input_ids)
        all_attn.append(attn)
        all_labels.append(labels)
    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attn,
        "labels": all_labels,
    })


train_ds = build_hf_dataset(train_df, tokenizer, CONFIG["MAX_LEN"])
val_ds = build_hf_dataset(val_df, tokenizer, CONFIG["MAX_LEN"])
test_ds = build_hf_dataset(test_df, tokenizer, CONFIG["MAX_LEN"])

print(train_ds)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17178
})


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels, pred_labels = [], []
    for pred_seq, lab_seq in zip(predictions, labels):
        cur_true, cur_pred = [], []
        for p, l in zip(pred_seq, lab_seq):
            if l == -100:
                continue
            cur_true.append(ID2LABEL[int(l)])
            cur_pred.append(ID2LABEL[int(p)])
        true_labels.append(cur_true)
        pred_labels.append(cur_pred)

    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
        "accuracy": (
            sum(len(set(t) & set(p)) for t, p in zip(true_labels, pred_labels)) /
            sum(len(set(t) | set(p)) for t, p in zip(true_labels, pred_labels))
            if sum(len(set(t) | set(p)) for t, p in zip(true_labels, pred_labels)) > 0 else 0.0
        ),
    }


import transformers as _tf
_tf_version = tuple(int(x) for x in _tf.__version__.split(".")[:2])
_eval_strategy_kwarg = (
    {"eval_strategy": "epoch"}
    if _tf_version >= (4, 41)
    else {"evaluation_strategy": "epoch"}
)

training_args = TrainingArguments(
    output_dir="train_output",
    num_train_epochs=CONFIG["EPOCHS"],
    per_device_train_batch_size=CONFIG["BATCH_SIZE"],
    per_device_eval_batch_size=CONFIG["EVAL_BATCH_SIZE"],
    gradient_accumulation_steps=CONFIG["GRADIENT_ACCUMULATION_STEPS"],
    learning_rate=CONFIG["LEARNING_RATE"],
    warmup_ratio=CONFIG["WARMUP_RATIO"],
    weight_decay=CONFIG["WEIGHT_DECAY"],
    optim="adamw_torch",
    **_eval_strategy_kwarg,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=2,
    report_to=[],
    seed=CONFIG["RANDOM_SEED"],
    fp16=CONFIG["FP16"],
    dataloader_num_workers=CONFIG["DATALOADER_NUM_WORKERS"],
    dataloader_pin_memory=torch.cuda.is_available(),
    group_by_length=True,
)

# [v9] Dùng WeightedTrainer thay Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["EARLY_STOPPING_PATIENCE"])],
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
train_result = trainer.train()
print(train_result)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.890731,0.868680,0.794351,0.982230,0.878356,0.952463
2,0.865898,0.850970,0.906649,0.985907,0.944618,0.981327
3,0.847608,0.843848,0.922813,0.988971,0.954747,0.978879
4,0.843710,0.843566,0.955516,0.987132,0.971067,0.982328
5,0.833794,0.841931,0.949618,0.989379,0.969091,0.979618
6,0.834054,0.841137,0.951493,0.989583,0.970164,0.978632
7,0.832650,0.841362,0.956178,0.989379,0.972495,0.981346
8,0.833467,0.840323,0.969363,0.988766,0.978969,0.986315
9,0.832432,0.840472,0.963044,0.989992,0.976332,0.985320
10,0.832885,0.840741,0.968419,0.989583,0.978887,0.986819


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2690, training_loss=0.8636283831968626, metrics={'train_runtime': 3119.7319, 'train_samples_per_second': 55.062, 'train_steps_per_second': 0.862, 'total_flos': 1.683223600488192e+16, 'train_loss': 0.8636283831968626, 'epoch': 10.0})


In [15]:
val_metrics = trainer.evaluate(val_ds)
print("Kết quả Validation:", val_metrics)

test_metrics = trainer.evaluate(test_ds)
print("Kết quả Test:", test_metrics)


Kết quả Validation: {'eval_loss': 0.8405597805976868, 'eval_precision': 0.9695573803324654, 'eval_recall': 0.9887663398692811, 'eval_f1': 0.9790676509252706, 'eval_accuracy': 0.9863152559553978, 'eval_runtime': 11.6895, 'eval_samples_per_second': 183.669, 'eval_steps_per_second': 1.454, 'epoch': 10.0}
Kết quả Test: {'eval_loss': 0.8474790453910828, 'eval_precision': 0.9636250259821243, 'eval_recall': 0.9908100021372088, 'eval_f1': 0.9770284510010537, 'eval_accuracy': 0.9861360718870347, 'eval_runtime': 11.8102, 'eval_samples_per_second': 181.877, 'eval_steps_per_second': 1.439, 'epoch': 10.0}


## 6. Giai đoạn 5: Lưu mô hình

Lưu model + tokenizer + config + label_mapping vào `models/visobert_teencode_detector/` để có thể load lại dự đoán mà không cần train lại.

In [16]:
model_dir = CONFIG["MODEL_DIR"]
os.makedirs(model_dir, exist_ok=True)

trainer.save_model(model_dir)          # lưu model + config
tokenizer.save_pretrained(model_dir)   # lưu tokenizer

label_mapping = {"label2id": LABEL2ID, "id2label": ID2LABEL}
with open(os.path.join(model_dir, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print("Đã lưu mô hình tại:", model_dir)
print("Nội dung thư mục:", os.listdir(model_dir))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình tại: /kaggle/working/models/phobert_teencode_detector
Nội dung thư mục: ['model.safetensors', 'label_mapping.json', 'config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json']


### Load lại mô hình (ví dụ sử dụng sau này, không cần train lại)
```python
loaded_tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_DIR"], use_fast=False)
loaded_model = AutoModelForTokenClassification.from_pretrained(CONFIG["MODEL_DIR"])
```

## 7. Giai đoạn 6: Đánh giá trên Dataset CFS (so với nhãn thật)

Luồng: `CFS text → Model ViSoBERT → Danh sách teencode dự đoán → So sánh với nhãn CFS (cột Nhãn (gốc))`

Đánh giá ở **mức cụm từ teencode trong câu** (so khớp tập hợp các teencode dự đoán với tập hợp teencode đã gán nhãn thật trong CFS), tính Precision / Recall / F1-score / Accuracy.

> ⚠️ **Vấn đề "lan nhãn" (label bleeding) và cách khắc phục:** Với các câu CFS viết rất nhiều teencode liên tiếp, mô hình (đặc biệt khi mới train ít epoch trên nhãn yếu) có thể nhận diện sai ranh giới thực thể — gán `I-TEENCODE` tràn sang cả các từ tiếng Việt bình thường nằm giữa, biến cả câu thành một cụm "teencode" dài bất thường. Hệ quả: (1) cụm dự đoán dài, vô nghĩa; (2) các teencode đã biết bên trong cụm đó (vd: `cmt`, `ib`, `size`) bị "nuốt" chung nên không khớp với từ điển → bị tính nhầm là "mới" ở Giai đoạn 8.
>
> **Cách khắc phục trong hàm `predict_teencode_spans` dưới đây:** nếu cụm dự đoán dài hơn `MAX_VALID_SPAN_LEN` (= độ dài mục từ điển dài nhất + 1), cụm đó được xem là khả năng cao bị lan nhãn → quét lại bằng chính từ điển teencode (n-gram match) để chỉ giữ lại phần khớp thật, phần còn lại bị loại bỏ. Đồng thời lọc bỏ các cụm có độ tin cậy trung bình thấp hơn `MIN_CONFIDENCE`. Cách này **không cần train lại model**, xử lý ngay ở bước hậu kiểm tra (post-processing).

In [17]:
def rescan_lexicon_match(span_tokens):
    """Quét lại 1 danh sách token bằng từ điển — dùng để cứu teencode trong
    cụm model dự đoán quá dài (lan nhãn).
    """
    n = len(span_tokens)
    matches = []
    i = 0
    while i < n:
        matched_len = 0
        max_try = min(MAX_LEX_LEN, n - i)
        for L in range(max_try, 0, -1):
            candidate = tuple(span_tokens[i:i + L])
            if candidate in LEXICON_BY_LEN.get(L, set()):
                matched_len = L
                break
        if matched_len > 0:
            phrase = " ".join(span_tokens[i:i + matched_len])
            matches.append(phrase.strip())
            i += matched_len
        else:
            i += 1
    return matches


B_ID = LABEL2ID["B-TEENCODE"]
I_ID = LABEL2ID["I-TEENCODE"]


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline: Dictionary Scan → BERT → Fusion
# ─────────────────────────────────────────────────────────────────────────────

def dictionary_scan(tokens):
    """Bước 1 — Dictionary Scan.
    Trả về candidates: {token_index: (length, group)}
      group = 'A': Nhóm A (an toàn, rescue được)
      group = 'B': Nhóm B (nhập nhằng, KHÔNG rescue)
    """
    n = len(tokens)
    candidates = {}
    i = 0
    while i < n:
        matched_len = 0
        matched_group = None
        max_try = min(MAX_LEX_LEN, n - i)
        for L in range(max_try, 0, -1):
            candidate = tuple(tokens[i:i + L])
            if candidate in LEXICON_A_BY_LEN.get(L, set()):
                matched_len = L
                matched_group = 'A'
                break
            elif candidate in LEXICON_BY_LEN.get(L, set()):
                matched_len = L
                matched_group = 'B'
                break
        if matched_len > 0:
            candidates[i] = (matched_len, matched_group)
            i += matched_len
        else:
            i += 1
    return candidates


def get_rescue_threshold(token_tuple, config):
    """[v9] Ngưỡng rescue động theo độ dài từ.
    Từ ngắn hơn = nhập nhằng hơn = cần ngưỡng cao hơn để rescue.
    Từ dài hơn = ít nhập nhằng = tin hơn khi dict match.
    """
    base = config.get("RESCUE_MIN_CONFIDENCE", 0.10)
    char_len = len(" ".join(token_tuple).replace(" ", ""))
    if char_len == 1:
        return base * 2.5    # 1 ký tự: x2.5 (e.g. 0.25)
    elif char_len == 2:
        return base * 1.5    # 2 ký tự: x1.5 (e.g. 0.15)
    elif char_len <= 4:
        return base          # 3-4 ký tự: base (e.g. 0.10)
    else:
        return base * 0.6    # >4 ký tự: x0.6 (e.g. 0.06)


def fusion_labels(tokens, word_pred_labels, word_probs_full, dict_candidates, config):
    """Bước 3 — Fusion.
    [v9] Dùng get_rescue_threshold() thay ngưỡng cứng.
    Kết hợp dict_candidates với word_pred_labels theo luật ưu tiên:
      Case 1: Nhóm A + BERT gán O + conf >= threshold → RESCUE
      Case 2: Nhóm B + BERT gán O → GIỮ O (bảo toàn Precision)
      Case 3: BERT gán B-/I- → tin BERT (phát hiện teencode mới)
    """
    fused_labels = list(word_pred_labels)

    for start, (length, group) in dict_candidates.items():
        end = start + length
        all_o = all(fused_labels[j] == "O" for j in range(start, end))
        if not all_o:
            continue  # BERT đã nhận ra → không can thiệp

        b_conf = float(word_probs_full[start][B_ID])
        i_confs = [float(word_probs_full[j][I_ID]) for j in range(start + 1, end)]
        span_conf = min([b_conf] + i_confs) if i_confs else b_conf

        if group == 'A':
            tok_tuple = tuple(tokens[start:end])
            threshold = get_rescue_threshold(tok_tuple, config)
            if span_conf >= threshold:
                fused_labels[start] = "B-TEENCODE"
                for j in range(start + 1, end):
                    fused_labels[j] = "I-TEENCODE"
        # group B → giữ O

    return fused_labels


def predict_teencode_spans(text, model, tokenizer, max_len, device, config=CONFIG):
    """[v9] Dự đoán teencode: Dictionary Scan → BERT → Fusion.
    [v9] min_conf hạ xuống dùng CONFIG["MIN_CONFIDENCE"] (0.4 thay 0.5).
    """
    tokens = tokenize(text)
    if len(tokens) == 0:
        return [], []

    dict_candidates = dictionary_scan(tokens)

    dummy_labels = ["O"] * len(tokens)
    input_ids, attn, _ = encode_with_labels(tokens, dummy_labels, tokenizer, max_len)
    input_ids_t = torch.tensor([input_ids]).to(device)
    attn_t = torch.tensor([attn]).to(device)

    # Unwrap DataParallel nếu cần (inference chỉ cần 1 GPU)
    infer_model = model.module if hasattr(model, "module") else model
    infer_model.eval()
    with torch.no_grad():
        outputs = infer_model(input_ids=input_ids_t, attention_mask=attn_t)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs = probs.cpu().numpy()

    word_pred_labels, word_confidences, word_probs_full = [], [], []
    pos = 1
    for word in tokens:
        sub_tokens = tokenizer.tokenize(word)
        if len(sub_tokens) == 0:
            sub_tokens = [tokenizer.unk_token]
        n_sub = len(sub_tokens)
        if pos < len(pred_ids):
            word_pred_labels.append(ID2LABEL[int(pred_ids[pos])])
            word_confidences.append(float(probs[pos][pred_ids[pos]]))
            word_probs_full.append(probs[pos])
        else:
            word_pred_labels.append("O")
            word_confidences.append(0.0)
            word_probs_full.append(np.zeros(len(CONFIG["LABELS"])))
        pos += n_sub

    fused_labels = fusion_labels(tokens, word_pred_labels, word_probs_full,
                                 dict_candidates, config)

    max_valid_len = config.get("MAX_VALID_SPAN_LEN", MAX_LEX_LEN + 1)
    min_conf = config.get("MIN_CONFIDENCE", 0.4)  # [v9] hạ từ 0.5 → 0.4

    spans = []
    noisy_raw_spans = []
    i = 0
    n = len(tokens)
    while i < n:
        if fused_labels[i] == "B-TEENCODE":
            j = i + 1
            while j < n and fused_labels[j] == "I-TEENCODE":
                j += 1
            span_len = j - i
            phrase_tokens = tokens[i:j]
            was_rescued = (word_pred_labels[i] == "O")
            if was_rescued:
                b_conf = float(word_probs_full[i][B_ID])
                i_confs = [float(word_probs_full[k][I_ID]) for k in range(i + 1, j)]
                conf = min([b_conf] + i_confs) if i_confs else b_conf
            else:
                conf = float(np.mean(word_confidences[i:j]))

            if span_len <= max_valid_len:
                if conf >= min_conf or was_rescued:
                    phrase_text = " ".join(phrase_tokens)
                    spans.append({
                        "text": phrase_text.strip(),
                        "confidence": round(conf, 4),
                        "source": "fusion" if was_rescued else "bert",
                    })
            else:
                noisy_raw_spans.append(" ".join(phrase_tokens).strip())
                for dict_phrase in rescan_lexicon_match(phrase_tokens):
                    spans.append({"text": dict_phrase, "confidence": 1.0, "source": "dict_rescan"})
            i = j
        else:
            i += 1

    return spans, noisy_raw_spans


# Demo
demo_text = "ai biết ko mn không ib rep mình nha"
demo_spans, demo_noisy = predict_teencode_spans(demo_text, model, tokenizer, CONFIG["MAX_LEN"], device)
print(f"Input : {demo_text}")
print(f"Spans : {demo_spans}")
print(f"Noisy : {demo_noisy}")


Input : ai biết ko mn không ib rep mình nha
Spans : [{'text': 'ko', 'confidence': 0.956, 'source': 'bert'}, {'text': 'mn', 'confidence': 0.9551, 'source': 'bert'}, {'text': 'ib', 'confidence': 0.9606, 'source': 'bert'}, {'text': 'rep', 'confidence': 0.9607, 'source': 'bert'}]
Noisy : []


In [18]:
def parse_gold_labels(s):
    '''Parse cột Nhãn (gốc) (dạng \"a\", \"b\", \"c\") thành list teencode đã chuẩn hóa.'''
    if not isinstance(s, str):
        return []
    items = re.findall(r'"([^"]+)"', s)
    return [unicodedata.normalize("NFC", it).strip().lower() for it in items]


# Lấy lại các câu CFS gốc tương ứng với test_df
# [fix] Build lookup dict từ cfs_sample để tra theo sent_id an toàn sau augment
cfs_lookup = cfs_sample.to_dict(orient='index')  # {index: row_dict}

eval_rows = []
for _, row in test_df.iterrows():
    sent_id = row["sent_id"]
    cfs_row = cfs_lookup.get(sent_id)
    if cfs_row is None:
        continue
    gold_col = cfs_row.get("Nhãn (gốc)")
    eval_rows.append({
        "text": cfs_row[cfs_text_col],
        "gold": parse_gold_labels(gold_col),
    })

eval_df = pd.DataFrame(eval_rows)
print(f"Số câu dùng để đánh giá (Test set, có gắn nhãn CFS thật): {len(eval_df)}")
eval_df.head(3)


Số câu dùng để đánh giá (Test set, có gắn nhãn CFS thật): 2148


,text,gold
0,muốn nhắn tin làm quen với Phúc Khanq nhma hon...,"[hong, nhma]"
1,Mọi người ơi học kì vừa rồi khoảng bao lâu mới...,[v]
2,Mng oi cho mình hỏi ở Lk chỗ nào bán thạch dừa...,"[lk, mụi ngừi, mng]"


In [19]:
# ── DÒ NGƯỠNG MIN_CONFIDENCE / RESCUE_MIN_CONFIDENCE TRÊN VALIDATION SET ──────
# (KHÔNG dò trực tiếp trên test set để tránh leakage / overfit ngưỡng vào test)
# Mục tiêu: Recall >= 75% trên validation, tối đa hoá Precision trong số các tổ hợp đạt mục tiêu.

# ── Helper functions (định nghĩa lại tại đây để cell này độc lập, không phụ thuộc
#    thứ tự chạy với cell đánh giá test phía dưới) ──────────────────────────────
def normalize_phrase(p):
    """Chuẩn hóa phrase để so sánh: chỉ lowercase + chuẩn hóa khoảng trắng,
    KHÔNG áp clean_text() vì clean_text loại ký tự đặc biệt sẽ làm méo
    các teencode như 'b1', 'in4', '@', ... và làm tăng FN giả."""
    if not isinstance(p, str):
        return ""
    return unicodedata.normalize("NFC", p).strip().lower()


def partial_match_score(pred_set, gold_set, threshold=2/3):
    """
    - Nếu cả pred và gold đều rỗng → đúng hoàn toàn (TP = 1, không có FP/FN)
    - Với mỗi term trong gold: nếu model dự đoán được >= threshold số từ của term → tính là TP
    - Với mỗi term trong pred: nếu >= threshold số từ của term khớp với gold → không tính FP
    Trả về: (tp_recall, fp, fn, exact_match)
    """
    if not pred_set and not gold_set:
        return 1, 0, 0, True

    def words(s):
        return s.strip().lower().split()

    def partial_covered(target, candidates, thr):
        t_words = words(target)
        if not t_words:
            return False
        for cand in candidates:
            c_words = words(cand)
            overlap = sum(1 for w in t_words if w in c_words)
            if overlap / len(t_words) >= thr:
                return True
        return False

    tp_recall = sum(1 for g in gold_set if partial_covered(g, pred_set, threshold))
    fn = len(gold_set) - tp_recall

    tp_prec = sum(1 for p in pred_set if partial_covered(p, gold_set, threshold))
    fp = len(pred_set) - tp_prec

    exact = (pred_set == gold_set)
    return tp_recall, fp, fn, exact


def build_eval_set(split_df):
    rows = []
    for _, row in split_df.iterrows():
        sent_id = row["sent_id"]
        cfs_row = cfs_lookup.get(sent_id)
        if cfs_row is None:
            continue
        gold_col = cfs_row.get("Nhãn (gốc)")
        rows.append({
            "text": cfs_row[cfs_text_col],
            "gold": parse_gold_labels(gold_col),
        })
    return pd.DataFrame(rows)

val_eval_df = build_eval_set(val_df)


def evaluate_with_thresholds(eval_df_, min_conf, rescue_min_conf):
    cfg = dict(CONFIG)
    cfg["MIN_CONFIDENCE"] = min_conf
    cfg["RESCUE_MIN_CONFIDENCE"] = rescue_min_conf
    TP_, FP_, FN_ = 0, 0, 0
    for _, row in eval_df_.iterrows():
        pred_spans, _ = predict_teencode_spans(row["text"], model, tokenizer, CONFIG["MAX_LEN"], device, config=cfg)
        pred_set = {normalize_phrase(s["text"]) for s in pred_spans if s["text"]}
        gold_set = {normalize_phrase(g) for g in row["gold"] if g}
        tp, fp, fn, _ = partial_match_score(pred_set, gold_set)
        TP_ += tp; FP_ += fp; FN_ += fn
    p = TP_ / (TP_ + FP_) if (TP_ + FP_) > 0 else 0.0
    r = TP_ / (TP_ + FN_) if (TP_ + FN_) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


TARGET_RECALL = 0.75
print(f"{'MIN_CONF':>9} {'RESCUE_CONF':>12} {'Precision':>10} {'Recall':>8} {'F1':>8}")
best = None
for min_conf in [0.5, 0.4, 0.3]:
    for rescue_conf in [min_conf, 0.3, 0.2, 0.15, 0.1, 0.05]:
        p, r, f1 = evaluate_with_thresholds(val_eval_df, min_conf, rescue_conf)
        print(f"{min_conf:>9} {rescue_conf:>12} {p:>10.2%} {r:>8.2%} {f1:>8.2%}")
        if r >= TARGET_RECALL:
            if best is None or p > best[0]:
                best = (p, r, f1, min_conf, rescue_conf)

if best:
    p, r, f1, min_conf, rescue_conf = best
    print(f"\n✅ Ngưỡng tốt nhất đạt Recall >= {TARGET_RECALL:.0%} (đo trên VALIDATION set):")
    print(f"   MIN_CONFIDENCE={min_conf} | RESCUE_MIN_CONFIDENCE={rescue_conf}")
    print(f"   -> Precision={p:.2%}, Recall={r:.2%}, F1={f1:.2%}")
    CONFIG["MIN_CONFIDENCE"] = min_conf
    CONFIG["RESCUE_MIN_CONFIDENCE"] = rescue_conf
else:
    print(f"\n⚠️ Không tổ hợp ngưỡng nào đạt Recall >= {TARGET_RECALL:.0%} chỉ bằng threshold/rescue.")
    print("   -> Phần lớn FN nằm NGOÀI lexicon (model chưa từng học) — cần mở rộng")
    print("      teencode_DATASET bằng các teencode mới (xem Giai đoạn 8/9) rồi TRAIN LẠI.")
    # Vẫn áp dụng tổ hợp tốt nhất theo F1 để không làm giảm chất lượng hiện tại
    best_f1 = max(
        ((evaluate_with_thresholds(val_eval_df, mc, rc), mc, rc)
         for mc in [0.5, 0.4, 0.3] for rc in [mc, 0.3, 0.2, 0.15, 0.1, 0.05]),
        key=lambda x: x[0][2]
    )
    (p, r, f1), min_conf, rescue_conf = best_f1
    CONFIG["MIN_CONFIDENCE"] = min_conf
    CONFIG["RESCUE_MIN_CONFIDENCE"] = rescue_conf

print("\nCONFIG đã cập nhật:", {k: CONFIG[k] for k in ["MIN_CONFIDENCE", "RESCUE_MIN_CONFIDENCE"]})


 MIN_CONF  RESCUE_CONF  Precision   Recall       F1
      0.5          0.5     80.63%   84.56%   82.55%
      0.5          0.3     80.62%   84.61%   82.57%
      0.5          0.2     80.62%   84.69%   82.61%
      0.5         0.15     80.63%   84.82%   82.67%
      0.5          0.1     80.68%   85.24%   82.90%
      0.5         0.05     80.68%   85.24%   82.90%
      0.4          0.4     80.41%   84.76%   82.53%
      0.4          0.3     80.42%   84.81%   82.56%
      0.4          0.2     80.42%   84.89%   82.60%
      0.4         0.15     80.43%   85.02%   82.66%
      0.4          0.1     80.48%   85.44%   82.89%
      0.4         0.05     80.48%   85.44%   82.89%
      0.3          0.3     80.41%   84.83%   82.56%
      0.3          0.3     80.41%   84.83%   82.56%
      0.3          0.2     80.41%   84.92%   82.60%
      0.3         0.15     80.41%   85.05%   82.66%
      0.3          0.1     80.47%   85.47%   82.89%
      0.3         0.05     80.47%   85.47%   82.89%

✅ Ngưỡng tố

In [20]:
# normalize_phrase() và partial_match_score() đã được định nghĩa ở cell dò
# ngưỡng MIN_CONFIDENCE/RESCUE_MIN_CONFIDENCE phía trên — dùng lại tại đây,
# không định nghĩa lại để tránh trùng lặp code.


# ── Hàm lấy các gold term thực sự bị bỏ sót (dùng partial match ngược) ────────
def get_missed_terms(gold_set, pred_set, threshold=2/3):
    """Trả về các gold term mà KHÔNG có pred nào cover được >= threshold."""
    def words(s):
        return s.strip().lower().split()
    def partial_covered(target, candidates, thr):
        t_words = words(target)
        if not t_words:
            return False
        for cand in candidates:
            c_words = words(cand)
            overlap = sum(1 for w in t_words if w in c_words)
            if overlap / len(t_words) >= thr:
                return True
        return False
    return {g for g in gold_set if not partial_covered(g, pred_set, threshold)}


TP, FP, FN = 0, 0, 0
exact_match_count = 0
both_empty_count = 0

fn_counter = Counter()
fn_context  = defaultdict(list)
fn_detail_rows = []       # chi tiết từng FN — dùng cho phân tích sau

KNOWN_NORM = {normalize_phrase(k) for k in KNOWN_TEENCODE_SET}

for _, row in eval_df.iterrows():
    pred_spans, noisy_spans = predict_teencode_spans(row["text"], model, tokenizer, CONFIG["MAX_LEN"], device)
    pred_set = {normalize_phrase(s["text"]) for s in pred_spans if s["text"]}
    gold_set = {normalize_phrase(g) for g in row["gold"] if g}

    tp, fp, fn, exact = partial_match_score(pred_set, gold_set)
    TP += tp
    FP += fp
    FN += fn

    if not pred_set and not gold_set:
        both_empty_count += 1

    # Thu thập FN term qua partial-match (chính xác hơn so với set difference)
    if gold_set:
        missed = get_missed_terms(gold_set, pred_set)
        for term in missed:
            fn_counter[term] += 1
            if len(fn_context[term]) < 5:
                fn_context[term].append(row["text"])
            fn_detail_rows.append({
                "fn_term": term,
                "text": row["text"],
                "gold_set": str(gold_set),
                "pred_set": str(pred_set),
                "noisy_spans": str(noisy_spans),
                "in_lexicon": term in KNOWN_NORM,
            })

    if exact:
        exact_match_count += 1

fn_df = pd.DataFrame(fn_detail_rows)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0

print(f"Số câu dùng để đánh giá (Test set, có gắn nhãn CFS thật): {len(eval_df)}")
print(f"📊 Đánh giá theo tiêu chí MỚI (partial match ≥2/3 từ, cả hai rỗng = đúng):")
print(f"Precision : {precision:.2%}")
print(f"Recall    : {recall:.2%}")
print(f"F1-score  : {f1:.2%}")
print(f"Accuracy  : {accuracy:.2%}  (TP / (TP+FP+FN), cùng cơ chế với Precision/Recall/F1)")
print(f"Câu cả hai đều rỗng (không có teencode): {both_empty_count}")
print()

TOP_FN = 30
print(f"{'='*60}")
print(f"🔴 TOP {TOP_FN} TEENCODE BỊ BỎ SÓT NHIỀU NHẤT (False Negatives)")
print(f"   -> Ưu tiên bổ sung/gán nhãn các từ này vào dataset để nâng Recall")
print(f"{'='*60}")
print(f"{'#':<4} {'Teencode (FN)':<25} {'Lần bỏ sót':>11}  Context mẫu")
print(f"{'-'*95}")
for rank, (term, cnt) in enumerate(fn_counter.most_common(TOP_FN), 1):
    ctx_sample  = fn_context[term][0] if fn_context[term] else ""
    ctx_preview = (ctx_sample[:60] + "...") if len(ctx_sample) > 60 else ctx_sample
    print(f"{rank:<4} {term:<25} {cnt:>11}  {ctx_preview}")

print()
print(f"Tổng FN term duy nhất (test set) : {len(fn_counter)}")
print(f"Tổng lượt FN                     : {FN}")
print(f"fn_df đã sẵn sàng: {len(fn_df)} dòng chi tiết FN")

# ════════════════════════════════════════════════════════════════════════
# [v11] Lưu kết quả ViSoBERT để bảng so sánh 3 thuật toán
# ════════════════════════════════════════════════════════════════════════
visobert_eval_result = {
    "model": "ViSoBERT",
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "min_conf": CONFIG["MIN_CONFIDENCE"],
    "rescue_conf": CONFIG["RESCUE_MIN_CONFIDENCE"],
    "max_len": CONFIG["MAX_LEN"],
}
print("\n[v11] Kết quả ViSoBERT đã lưu để so sánh liên thuật toán:")
print(f"  Precision={precision:.2%}  Recall={recall:.2%}  F1={f1:.2%}")


Số câu dùng để đánh giá (Test set, có gắn nhãn CFS thật): 2148
📊 Đánh giá theo tiêu chí MỚI (partial match ≥2/3 từ, cả hai rỗng = đúng):
Precision : 82.29%
Recall    : 85.14%
F1-score  : 83.69%
Accuracy  : 71.96%  (TP / (TP+FP+FN), cùng cơ chế với Precision/Recall/F1)
Câu cả hai đều rỗng (không có teencode): 417

🔴 TOP 30 TEENCODE BỊ BỎ SÓT NHIỀU NHẤT (False Negatives)
   -> Ưu tiên bổ sung/gán nhãn các từ này vào dataset để nâng Recall
#    Teencode (FN)              Lần bỏ sót  Context mẫu
-----------------------------------------------------------------------------------------------
1    h                                  15  HOT Mắt kính Anna dành 100 suất thu kính cũ, đổi sang kính m...
2    mn                                 11  Có nhóm bóng chuyền sinh viên nào không cho mình ké với mn. ...
3    tr                                  9  vi 7i1 ơi, có crush hay ngyeu chưa, hôm trc thấy đi cùng bạn...
4    p                                   7  Bạn P. My 7i1 thích bạn Hữu Phước 7i2 đ

## 7b. Phân tích False Negative (FN) — Teencode bị bỏ sót

Phần này khai thác `fn_df` (đã tạo ở cell trên) để:
1. Thống kê các FN xuất hiện nhiều nhất
2. Phân loại FN: có trong lexicon (lỗi model) vs không có trong lexicon (từ điển thiếu)
3. Xem câu ví dụ cụ thể
4. Xuất file `FN_analysis.xlsx`


In [21]:
# ── 1 & 2. Tần suất + phân loại FN ──────────────────────────────────────────
if fn_df.empty:
    print("Không có FN nào trong test set — model dự đoán tốt!")
    fn_freq = pd.DataFrame(columns=["fn_term","count","in_lexicon"])
    fn_in_lex = fn_out_lex = fn_freq
else:
    fn_freq = fn_df["fn_term"].value_counts().reset_index()
    fn_freq.columns = ["fn_term", "count"]
    fn_freq["in_lexicon"] = fn_freq["fn_term"].apply(lambda t: t in KNOWN_NORM)

    fn_in_lex  = fn_freq[fn_freq["in_lexicon"] == True]
    fn_out_lex = fn_freq[fn_freq["in_lexicon"] == False]

    print(f"=== Top 30 teencode bị bỏ sót nhiều nhất ===")
    print(fn_freq.head(30).to_string(index=False))
    print(f"\n--- Phân loại ---")
    print(f"FN có trong lexicon (lỗi model):  {len(fn_in_lex)} term, {fn_in_lex['count'].sum()} lần")
    print(f"FN không trong lexicon (từ điển thiếu): {len(fn_out_lex)} term, {fn_out_lex['count'].sum()} lần")


=== Top 30 teencode bị bỏ sót nhiều nhất ===
fn_term  count  in_lexicon
      h     15        True
     mn     11        True
     tr      9        True
      p      7        True
    mìn      6       False
   huhu      6        True
      r      6        True
    hic      5       False
      g      5        True
      z      4        True
   k dc      4        True
   thui      4        True
     xl      4        True
     oi      4       False
     ib      4        True
      e      4        True
    mng      4        True
     đu      4       False
   hong      4        True
   ptnk      3        True
     kg      3        True
    cmt      3        True
  ngyeu      3       False
  hongg      3       False
   hđrl      3       False
   cccd      3       False
     ng      3        True
     ui      3        True
    crs      3       False
      n      3        True

--- Phân loại ---
FN có trong lexicon (lỗi model):  108 term, 212 lần
FN không trong lexicon (từ điển thiếu): 389 ter

In [22]:
# ── 3. Câu ví dụ cho từng FN term (top 20) ──────────────────────────────────
if fn_df.empty or fn_freq.empty:
    print("Không có dữ liệu FN để hiển thị.")
else:
    print("=== Câu ví dụ chứa các FN term bị bỏ sót nhiều nhất ===\n")
    for term in fn_freq["fn_term"].head(20):
        examples = fn_df[fn_df["fn_term"] == term]["text"].head(3).tolist()
        in_lex = term in KNOWN_NORM
        count_val = fn_freq.loc[fn_freq["fn_term"] == term, "count"].values[0]
        lex_label = "✓ lexicon" if in_lex else "✗ ngoài lexicon"
        print(f'[{lex_label}]  "{term}"  (bỏ sót {count_val} lần)')
        for ex in examples:
            print(f"   → {ex[:120]}")
        print()


=== Câu ví dụ chứa các FN term bị bỏ sót nhiều nhất ===

[✓ lexicon]  "h"  (bỏ sót 15 lần)
   → HOT Mắt kính Anna dành 100 suất thu kính cũ, đổi sang kính mới cho các bạn trong tên có chữ N,H,G. Dự kiến hết rất nhanh
   → Hôm thứ 2 em có để quên quyển sách, sổ ghi chép và cục sạc dự phòng. Em đã nhờ bạn len lấy lúc 3h chiều thì thấy cửa kho
   → 9h đki hp, 12h30 dậy lướt fb, lướt trúng bài mấy b đki hp mới chợt nhận ra...

[✓ lexicon]  "mn"  (bỏ sót 11 lần)
   → Có nhóm bóng chuyền sinh viên nào không cho mình ké với mn. Mình có chơi 1 thời gian mà nghỉ lâu r nên định chơi lại ấy
   → trường mình có cặp nào quen khác cơ sở không mn, tui muốn nghe review
   → Lịch sử Đảng thì ôn trọng tâm những bài nào vậy mn. Ôn tập trên Lms thì có khả năng ra trúng trong đó ko ạ.

[✓ lexicon]  "tr"  (bỏ sót 9 lần)
   → vi 7i1 ơi, có crush hay ngyeu chưa, hôm trc thấy đi cùng bạn mà mấy bạn gọi vi ơi xinh quá tr, xin in4 nữa
   → Em đóng trước 5tr rồi em gia hạn học phí đến ngày 3012026 đk vậy mn tổng 

In [23]:
# ── 4. Xuất file FN_analysis.xlsx ────────────────────────────────────────────
if fn_df.empty or fn_freq.empty:
    print("Không có FN → bỏ qua xuất file FN_analysis.xlsx")
else:
    fn_stats = fn_freq.copy()
    fn_stats["gợi ý xử lý"] = fn_stats.apply(
        lambda r: "Xem lại fine-tune / augment data" if r["in_lexicon"]
                  else "Cân nhắc bổ sung vào từ điển",
        axis=1
    )

    fn_detail_export = fn_df[["fn_term","text","gold_set","pred_set","noisy_spans","in_lexicon"]].copy()
    fn_detail_export.columns = ["FN term","Câu gốc","Nhãn thật (gold)","Model dự đoán (pred)","Noisy span (nếu có)","Có trong lexicon"]

    fn_export_path = os.path.join(CONFIG["OUTPUT_DIR"], "FN_analysis.xlsx")
    with pd.ExcelWriter(fn_export_path, engine="openpyxl") as writer:
        fn_stats.to_excel(writer, sheet_name="Thống kê FN", index=False)
        fn_detail_export.to_excel(writer, sheet_name="Chi tiết FN", index=False)

    print(f"Đã lưu phân tích FN tại: {fn_export_path}")
    print(f"  Sheet \'Thống kê FN\'  : {len(fn_stats)} term")
    print(f"  Sheet \'Chi tiết FN\'  : {len(fn_detail_export)} dòng")

    pct_in_lex  = len(fn_in_lex)  / len(fn_freq) * 100 if len(fn_freq) > 0 else 0
    pct_out_lex = len(fn_out_lex) / len(fn_freq) * 100 if len(fn_freq) > 0 else 0

    print("\n========== GỢI Ý NÂNG CẤP MÔ HÌNH DỰA TRÊN FN ==========")
    print(f"  {pct_in_lex:.1f}% FN term đã có trong lexicon")
    print( "    → Hướng xử lý: tăng EPOCHS, giảm MIN_CONFIDENCE, data augmentation")
    print(f"  {pct_out_lex:.1f}% FN term KHÔNG có trong lexicon")
    print( "    → Hướng xử lý: bổ sung vào từ điển rồi chạy lại từ Giai đoạn 3")
    print("=============================================================")


Đã lưu phân tích FN tại: /kaggle/working/outputs/FN_analysis.xlsx
  Sheet 'Thống kê FN'  : 497 term
  Sheet 'Chi tiết FN'  : 673 dòng

========== GỢI Ý NÂNG CẤP MÔ HÌNH DỰA TRÊN FN ==========
  21.7% FN term đã có trong lexicon
    → Hướng xử lý: tăng EPOCHS, giảm MIN_CONFIDENCE, data augmentation
  78.3% FN term KHÔNG có trong lexicon
    → Hướng xử lý: bổ sung vào từ điển rồi chạy lại từ Giai đoạn 3


## 8. Giai đoạn 7: Chạy model trên toàn bộ CFS

Output: file `CFS_teencode_prediction.xlsx` **giữ nguyên toàn bộ các cột của file CFS đầu vào** (đứng trước, đúng thứ tự gốc), sau đó thêm 3 cột mới ở cuối: `Teencode (nhãn thật - CFS)`, `Teencode (ViSoBERT phát hiện)` (2 cột này đặt **sát nhau** để dễ so sánh bằng mắt) và `Confidence`.

> Theo cấu hình `CFS_SAMPLE_SIZE = 5000`, bước này chạy trên mẫu 5.000 dòng CFS (đặt `CFS_SAMPLE_SIZE = None` ở Mục 1 để chạy full ~30.000 dòng).

In [24]:
run_df = valid_cfs if CONFIG["CFS_SAMPLE_SIZE"] is None else valid_cfs.sample(
    n=min(CONFIG["CFS_SAMPLE_SIZE"], len(valid_cfs)), random_state=CONFIG["RANDOM_SEED"]
)
run_df = run_df.reset_index(drop=True)
print(f"Số dòng CFS sẽ chạy dự đoán: {len(run_df)}")

# Giữ lại đúng thứ tự + toàn bộ tên cột gốc của file CFS đầu vào -> các cột này sẽ đứng TRƯỚC trong file kết quả
original_cols = list(run_df.columns)
has_gold_col = "Nhãn (gốc)" in run_df.columns

prediction_records = []
noisy_span_counter = Counter()  # đếm các cụm RAW bị model lan nhãn quá dài (để theo dõi chất lượng model)

for _, row in run_df.iterrows():
    text = row[cfs_text_col]
    spans, noisy = predict_teencode_spans(text, model, tokenizer, CONFIG["MAX_LEN"], device)
    teencode_list = [s["text"] for s in spans]
    confidences = [s["confidence"] for s in spans]

    # Nhãn thật lấy từ cột Nhãn (gốc) của CFS (nếu có) -> parse giống Giai đoạn 6
    gold_list = parse_gold_labels(row.get("Nhãn (gốc)")) if has_gold_col else []

    record = {col: row[col] for col in original_cols}      # 1) toàn bộ thông tin gốc của file CFS, đứng trước
    record.update({                                          # 2) thông tin mới của ViSoBERT, 2 cột nhãn đặt SÁT NHAU để dễ so sánh
        "Teencode (nhãn thật - CFS)": ", ".join(gold_list),
        "Teencode (ViSoBERT phát hiện)": ", ".join(teencode_list),
        "Confidence": ", ".join(f"{c:.2f}" for c in confidences),
    })
    prediction_records.append(record)
    for raw in noisy:
        noisy_span_counter[raw] += 1

pred_result_df = pd.DataFrame(prediction_records)

print(f"Số cụm bị model lan nhãn quá dài (đã hậu kiểm tra lại bằng từ điển): {sum(noisy_span_counter.values())}")
if noisy_span_counter:
    print("Ví dụ một vài cụm bị lan nhãn (để bạn theo dõi chất lượng model):")
    for raw, c in noisy_span_counter.most_common(5):
        print(f"  - \"{raw}\"  (x{c})")

pred_result_df.head(5)


Số dòng CFS sẽ chạy dự đoán: 21473
Số cụm bị model lan nhãn quá dài (đã hậu kiểm tra lại bằng từ điển): 0


,STT,Văn bản gốc (CFS),Độ dài (ký tự),Độ dài (từ),Nhãn (gốc),Teencode (nhãn thật - CFS),Teencode (ViSoBERT phát hiện),Confidence
0,1,UTC2 đúng thật sự là Đường đến thành công,41,9,"""UTC2""",utc2,utc2,0.97
1,2,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,39,10,"""KTX""",ktx,ktx,0.96
2,3,21 tuổi là lớn rồi nha Hậu trường tối qua,41,10,NaN,,,
3,4,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,42,10,"""UTC2""",utc2,utc2,0.97
4,5,Một đội hình hướng tới sự khéo ăn khéo nói,42,10,NaN,,,


In [25]:
output_path = os.path.join(CONFIG["OUTPUT_DIR"], "CFS_teencode_prediction.xlsx")
pred_result_df.to_excel(output_path, index=False)
print("Đã lưu:", output_path)


Đã lưu: /kaggle/working/outputs/CFS_teencode_prediction.xlsx


## 9. Giai đoạn 8: Khai phá teencode mới

```
Teencode mới = Teencode tìm được từ CFS  −  Teencode đã có trong dataset hiện tại
```

So sánh tập teencode model phát hiện được trên CFS (`pred_result_df`) với tập teencode đã có trong từ điển (`KNOWN_TEENCODE_SET`, xây ở Mục 4) để tìm ra các teencode **chưa từng có trong dataset**.

In [26]:
# Thu thập toàn bộ teencode model phát hiện được trên CFS + lưu lại câu chứa nó (context) + đếm số lần xuất hiện
detected_counter = Counter()
detected_context = defaultdict(list)

# Thu thập FN trên toàn bộ CFS có nhãn vàng (dùng để nâng cấp dataset)
cfs_fn_counter = Counter()
cfs_fn_context  = defaultdict(list)
has_gold_col_run = "Nhãn (gốc)" in pred_result_df.columns

for _, row in pred_result_df.iterrows():
    sent = row[cfs_text_col]
    phrases = [p.strip() for p in row["Teencode (ViSoBERT phát hiện)"].split(",") if p.strip()]
    for p in phrases:
        norm_p = normalize_phrase(p)
        if not norm_p:
            continue
        detected_counter[norm_p] += 1
        if len(detected_context[norm_p]) < 5:   # giữ tối đa 5 câu ví dụ / teencode
            detected_context[norm_p].append(sent)

    # FN trên CFS: so nhãn vàng vs dự đoán
    if has_gold_col_run:
        gold_list_run = parse_gold_labels(row.get("Nhãn (gốc)"))
        pred_list_run = [p.strip() for p in row["Teencode (ViSoBERT phát hiện)"].split(",") if p.strip()]
        gold_set_run = {normalize_phrase(g) for g in gold_list_run if g}
        pred_set_run = {normalize_phrase(p) for p in pred_list_run if p}
        for term in gold_set_run - pred_set_run:
            cfs_fn_counter[term] += 1
            if len(cfs_fn_context[term]) < 5:
                cfs_fn_context[term].append(sent)

print(f"Tổng số teencode (duy nhất) model phát hiện được trên mẫu CFS: {len(detected_counter)}")

# Teencode mới = phát hiện được trên CFS NHƯNG chưa có trong từ điển hiện tại
KNOWN_NORM = {normalize_phrase(k) for k in KNOWN_TEENCODE_SET}
new_teencode = {t: c for t, c in detected_counter.items() if t not in KNOWN_NORM}

print(f"Số teencode MỚI (chưa có trong dataset): {len(new_teencode)}")

# ── In Top-30 FN trên toàn CFS ────────────────────────────────────────────────
if has_gold_col_run and cfs_fn_counter:
    TOP_FN_CFS = 30
    print(f"\n{'='*60}")
    print(f"🔴 TOP {TOP_FN_CFS} FN TRÊN TOÀN BỘ CFS (nhãn vàng vs dự đoán)")
    print(f"   -> Đây là các teencode có nhãn thật nhưng model bỏ sót nhiều nhất")
    print(f"   -> Ưu tiên cao nhất để gán nhãn lại / bổ sung vào dataset")
    print(f"{'='*60}")
    print(f"{'#':<4} {'Teencode (FN)':<25} {'Lần bỏ sót':>11}  Context mẫu")
    print(f"{'-'*95}")
    for rank, (term, cnt) in enumerate(cfs_fn_counter.most_common(TOP_FN_CFS), 1):
        ctx_sample  = cfs_fn_context[term][0] if cfs_fn_context[term] else ""
        ctx_preview = (ctx_sample[:60] + "...") if len(ctx_sample) > 60 else ctx_sample
        print(f"{rank:<4} {term:<25} {cnt:>11}  {ctx_preview}")
    print(f"\nTổng FN term duy nhất (toàn CFS): {len(cfs_fn_counter)}")

# Xem 10 teencode mới xuất hiện nhiều nhất
print("\n10 teencode mới xuất hiện nhiều nhất:")
print(sorted(new_teencode.items(), key=lambda x: -x[1])[:10])


Tổng số teencode (duy nhất) model phát hiện được trên mẫu CFS: 1534
Số teencode MỚI (chưa có trong dataset): 577

🔴 TOP 30 FN TRÊN TOÀN BỘ CFS (nhãn vàng vs dự đoán)
   -> Đây là các teencode có nhãn thật nhưng model bỏ sót nhiều nhất
   -> Ưu tiên cao nhất để gán nhãn lại / bổ sung vào dataset
#    Teencode (FN)              Lần bỏ sót  Context mẫu
-----------------------------------------------------------------------------------------------
1    h                                 148  Tìm đội gl bóng đá sân tdc 5h chiều nay
2    p                                  63  Ráp súng tầm 1p hơn thì được bao nhiêu điểm ạ
3    đ                                  58  có ai biết cách săn đơn 0đ shoppe không ạ
4    tr                                 56  Tìm phòng 4tr 1ng ở, gần đại học hồng bàng ạ
5    cccd                               45  có bạn nào làm mất cccd lại xe hola tacos lấy nhe ạ
6    mn                                 44  tiếng anh giao tiếp 1 ôn gì để thi vậy mn.
7    hongg          

## 10. Giai đoạn 9: Xuất dataset mở rộng

Tạo file `teencode_missing_from_CFS.xlsx` gồm: `Teencode mới`, `Số lần xuất hiện`, `Context` (câu ví dụ chứa teencode đó) — dùng để **gán nhãn thủ công** và **bổ sung vào dataset** cho lần train tiếp theo.

In [27]:
export_records = []
for term, count in sorted(new_teencode.items(), key=lambda x: -x[1]):
    contexts = detected_context.get(term, [])
    export_records.append({
        "Teencode mới": term,
        "Số lần xuất hiện": count,
        "Nguồn": "model_detected_new",
        "Context": " | ".join(contexts),
    })

new_teencode_df = pd.DataFrame(export_records)
print(f"Số dòng trong dataset mở rộng (teencode mới): {len(new_teencode_df)}")
new_teencode_df.head(10)


Số dòng trong dataset mở rộng (teencode mới): 577


,Teencode mới,Số lần xuất hiện,Nguồn,Context
0,mn.,50,model_detected_new,tiếng anh giao tiếp 1 ôn gì để thi vậy mn. | t...
1,nè.,34,model_detected_new,Bạn nữ nào muốn đánh giải lq nữ ko lập team đá...
2,huhu.,30,model_detected_new,huhu mình tìm CLB bán móc khoá này ạaa. mê quá...
3,m.n,29,model_detected_new,Làm lại thẻ sv là tầng 6 hay 5 v m.n | chiều n...
4,mng.,24,model_detected_new,Năm nay UEH xét tuyển như thế nào vậy ạ mng. |...
5,í.,24,model_detected_new,Ý là bà V Tr bị phiền í. Đx sang lớp t nx | cá...
6,ko.,22,model_detected_new,Có ai 2k5 giờ mới tsv như tui ko. Sợ bị ngại |...
7,a.,17,model_detected_new,Có ai từn bị cr thic bạn thân chua a.. | Em cầ...
8,hong.,14,model_detected_new,Có ai thi vstep tháng 4 hong. Đăng kí chung ch...
9,đc.,13,model_detected_new,Cho em hỏi về trung tâm học tiếng anh đầu ra ổ...


In [28]:
new_teencode_path = os.path.join(CONFIG["OUTPUT_DIR"], "teencode_missing_from_CFS.xlsx")

with pd.ExcelWriter(new_teencode_path, engine="openpyxl") as writer:
    # Sheet 1: Teencode mới (model phát hiện, chưa có trong từ điển)
    new_teencode_df.to_excel(writer, sheet_name="Teencode_moi", index=False)

    # Sheet 2: Top FN từ test set (để ưu tiên gán nhãn lại)
    if fn_counter:
        fn_test_records = []
        for term, cnt in fn_counter.most_common():
            fn_test_records.append({
                "Teencode (FN)": term,
                "Số lần bỏ sót (test)": cnt,
                "Nguồn": "fn_test_set",
                "Context": " | ".join(fn_context.get(term, [])),
            })
        pd.DataFrame(fn_test_records).to_excel(writer, sheet_name="FN_test_set", index=False)

    # Sheet 3: Top FN từ toàn bộ CFS có nhãn vàng (ưu tiên cao nhất)
    if "cfs_fn_counter" in dir() and cfs_fn_counter:
        fn_cfs_records = []
        for term, cnt in cfs_fn_counter.most_common():
            fn_cfs_records.append({
                "Teencode (FN)": term,
                "Số lần bỏ sót (CFS)": cnt,
                "Nguồn": "fn_toan_bo_CFS",
                "Context": " | ".join(cfs_fn_context.get(term, [])),
            })
        pd.DataFrame(fn_cfs_records).to_excel(writer, sheet_name="FN_toan_bo_CFS", index=False)

print("Đã lưu:", new_teencode_path)
print("  Sheet 'Teencode_moi'   : teencode model tìm được chưa có trong từ điển")
print("  Sheet 'FN_test_set'    : FN từ test set (ưu tiên gán nhãn lại)")
if "cfs_fn_counter" in dir() and cfs_fn_counter:
    print("  Sheet 'FN_toan_bo_CFS' : FN trên toàn CFS có nhãn vàng (ưu tiên cao nhất)")


Đã lưu: /kaggle/working/outputs/teencode_missing_from_CFS.xlsx
  Sheet 'Teencode_moi'   : teencode model tìm được chưa có trong từ điển
  Sheet 'FN_test_set'    : FN từ test set (ưu tiên gán nhãn lại)
  Sheet 'FN_toan_bo_CFS' : FN trên toàn CFS có nhãn vàng (ưu tiên cao nhất)


## ✅ Tổng kết

Notebook đã thực hiện đầy đủ:

| Giai đoạn | Nội dung | Output |
|---|---|---|
| 2 | Làm sạch text + Tokenization (tách âm tiết) | hàm `clean_text`, `tokenize` |
| 3 | Xây dataset BIO (B-TEENCODE / I-TEENCODE / O) | `train_df`, `val_df`, `test_df` (80/10/10) |
| 4 | Train ViSoBERT Token Classification | mô hình fine-tuned trong `trainer` |
| 5 | Lưu mô hình | `models/visobert_teencode_detector/` (model, tokenizer, config, label_mapping.json) |
| 6 | Đánh giá trên CFS (so với nhãn thật) | Precision / Recall / F1 / Accuracy + Top FN test set |
| 7 | Chạy trên CFS (mẫu 5.000 dòng) | `outputs/CFS_teencode_prediction.xlsx` |
| 8 | Khai phá teencode mới + Top FN toàn CFS | tập `new_teencode`, `cfs_fn_counter` |
| 9 | Xuất dataset mở rộng + FN analysis | `outputs/teencode_missing_from_CFS.xlsx` (3 sheets: Teencode_moi, FN_test_set, FN_toan_bo_CFS) |

**Gợi ý bước tiếp theo:**
- Gán nhãn thủ công cho file `teencode_missing_from_CFS.xlsx`, bổ sung vào `teencode_DATASET_FINAL_v4_đã_điền.xlsx` rồi chạy lại từ Giai đoạn 3 để train phiên bản tiếp theo (vòng lặp **active learning**).
- Nếu cần độ chính xác cao hơn ở Giai đoạn 3, có thể bổ sung một vòng rà soát thủ công cho nhãn weak-label trước khi train, đặc biệt với các từ ngắn đa nghĩa (vd: "ai", "an", "y").
- Tăng `CFS_SAMPLE_SIZE` lên `None` (toàn bộ ~30.000 dòng) và `EPOCHS` lên 8-10 khi chạy trên GPU để có kết quả production-ready.
